# Prediction of Atmospheric Carbon Dioxide (CO₂)

Monthly mean CO₂ concentrations recorded at the [Mauna Loa Observatory, Hawaiʻi](https://gml.noaa.gov/ccgg/trends/data.html) — the longest continuous atmospheric CO₂ record in the world.

**Notebook outline**
1. Data loading and preprocessing
2. Decomposition: trend / seasonality / noise
3. Forecasting as a regression problem (linear trend → seasonal features)
4. Cross-validation with `TimeSeriesSplit`
5. SARIMA: automatic order selection with `pmdarima`
6. Prophet
7. Comparing all models

---
*Authors: Thomas Moreau, Mathurin Massias, Alexandre Gramfort*

In [ ]:
import numpy as np
import pandas as pd
from sklearn import linear_model

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Data loading *(don't read!)*

In [ ]:
data_full = pd.read_csv(
    "data/co2_mm_mlo.txt",
    comment="#", sep=r'\s+', index_col=False,
    usecols=[0, 1, 3], names=["year", "month", "co2"]
)
month_to_int = dict(january=1, february=2, march=3, april=4, may=5, june=6, july=7, august=8,
                    september=9, october=10, november=11, december=12)
int_to_month = {v: k for k, v in month_to_int.items()}
data_full["month"] = data_full["month"].apply(lambda m: int_to_month[m])
data = data_full.query("year > 2008").copy()
data = data.reset_index().drop("index", axis=1)
data.head()

In [ ]:
data.month

<div class="alert alert-success">
    <b>Questions:</b>
     <ul>
      <li>what is X? what is y?</li>
      <li>how would you evaluate your forecasting model?</li>
    </ul>
</div>

### Encode month as an integer

In [ ]:
data["month"] = data["month"].apply(lambda m: month_to_int[m] - 1).astype(int)
data.head()

In [ ]:
data.dtypes

In [ ]:
data['time'] = data['year'] + data['month'] / 12.

In [ ]:
data.time

In [ ]:
data.plot(x='time', y='co2');  # visualize the data

In [ ]:
data_train = data[data["year"] < 2018]  # take history as training data
data_test = data[data["year"] >= 2018]  # rest will be testing data
data_train.plot(x='time', y='co2');

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(data_train["time"], data_train["co2"], label="Train (< 2018)", color="steelblue")
ax.plot(data_test["time"],  data_test["co2"],  label="Test (≥ 2018)",  color="tomato")
ax.axvline(2018, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Year")
ax.set_ylabel("CO₂ (ppm)")
ax.set_title("Train / test split")
ax.legend()
plt.tight_layout()

## 2. Decomposition: trend + seasonality + noise

Before building any model it's worth decomposing the series into its components. This motivates the feature engineering we'll do next.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

seasonal_results = seasonal_decompose(data["co2"], period=12, model='add')

with plt.rc_context({'figure.figsize': (10, 6)}):
    seasonal_results.plot()

## 3. Forecasting as a regression problem

We frame CO₂ prediction as a standard regression: features are derived from the calendar (`year`, `month`), and the target is the CO₂ value. The model is trained on data before 2018 and evaluated on the held-out test set.

In [ ]:
X = data["time"].values[:, np.newaxis]
X_train = data_train["time"].values[:, np.newaxis]
y_train = data_train["co2"]

model = linear_model.LinearRegression()
model.fit(X_train, y_train)

plt.plot(data_train["time"], data_train["co2"]);
plt.plot(X, model.predict(X))
plt.xlabel('X (time)')
plt.ylabel('y (co2)');

Fit does not capture seasonality.

**How do you encode a cyclic variable like month?**

`month` takes values 1–12 where 12 and 1 are *adjacent* (December → January), but treating it as a plain integer makes the model think 12 and 1 are far apart. The fix: project onto the unit circle with $\cos$ and $\sin$. Two features together encode both *position* and *continuity* across the cycle:

$$\text{month\_cos} = \cos\!\left(\frac{2\pi \cdot \text{month}}{12}\right), \qquad \text{month\_sin} = \sin\!\left(\frac{2\pi \cdot \text{month}}{12}\right)$$

The full model becomes:

$$CO_2 = \theta_1 \cdot \text{year} + \theta_2 \cdot \cos\!\left(\frac{2\pi \cdot \text{month}}{12}\right) + \theta_3 \cdot \sin\!\left(\frac{2\pi \cdot \text{month}}{12}\right) + \theta_0$$

This trick works for any cyclic feature: hour of day, day of week, angle, etc.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

def add_cyclic_month(df):
    """Cyclic encoding of month (1–12) as cos/sin features.
    The pipeline calls this on a DataFrame with 'year' and 'month' columns.
    """
    return np.stack([
        df['year'],
        np.cos(df['month'] / 12. * 2 * np.pi),
        np.sin(df['month'] / 12. * 2 * np.pi),
    ], axis=1)

pipe = Pipeline([
    ('cyclic_features', FunctionTransformer(add_cyclic_month)),
    ('model', linear_model.LinearRegression()),
])

# Raw features — the pipeline handles all feature engineering internally
X = data[['year', 'month']]
X_train = data_train[['year', 'month']]
y_train = data_train['co2']

pipe.fit(X_train, y_train)

plt.plot(data_train["time"], data_train["co2"])
plt.plot(data["time"], pipe.predict(X));

In [ ]:
y = data['co2']
y

## 4. Cross-validation with TimeSeriesSplit

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

In [ ]:
TimeSeriesSplit?

In [ ]:
from sklearn.model_selection import cross_val_score

cv = TimeSeriesSplit()
scores = cross_val_score(pipe, X, y, cv=cv)
scores

In [ ]:
max_train_size = 36
cv = TimeSeriesSplit(n_splits=5, max_train_size=max_train_size, test_size=12)
scores = cross_val_score(pipe, X, y, cv=cv)
print(f"{max_train_size=}, Mean: {scores.mean(): .4f}  -- STD: {scores.std(): .4f}")

<div class="alert alert-success">
    <b>EXERCISE:</b>
     <ul>
      <li>Evaluate the impact of the max_train_size parameter.
          How many months are enough to predict reasonably well?</li>
    </ul>
</div>

Solution: `solutions/co2-1_model-selection-time-series-split.py`

<div class="alert alert-success">
    <b>EXERCISE:</b>
     <ul>
      <li>Use statsmodels to do the forecasting using an AR model.</li>
    </ul>
</div>

Solution: `solutions/co2-2_forecasting-AR-statsmodels.py`

## 5. SARIMA: automatic order selection

**Stationarity.** Classical time series models (AR, ARMA, SARIMA) assume the series is *stationary* — its mean and variance don't change over time. CO₂ has a clear upward trend, so it is not stationary. The `d` parameter in ARIMA handles this by differencing the series `d` times (subtracting consecutive values) until stationarity is achieved. `auto_arima` selects `d` automatically, typically `d=1` here.

We then use `pmdarima.auto_arima` to search over the full $(p,d,q)(P,D,Q)_m$ parameter space, and refit the chosen model with `statsmodels.SARIMAX` for coefficient estimates and AIC/BIC.

In [ ]:
# !pip install -U pmdarima

In [ ]:
from pmdarima import auto_arima

# Automatic determination of sarima parameters
sarima_params_res = auto_arima(
    data_train['co2'],
    seasonal=True,  # take into account the seasonality
    m=12  # the length of our cycle
)

# sarima_params_res.summary()

# Get the parameters:
arima_params = sarima_params_res.order
seasonal_params = sarima_params_res.seasonal_order

print(f'ARIMA parameters (p,d,q) = {arima_params}')
print(f'seasonal parameters (P,D,Q,m) =  {seasonal_params}')

In [ ]:
# model fitting using parameters fitted on the train data
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_model = SARIMAX(data_train['co2'],
                       order=arima_params,
                       seasonal_order=seasonal_params)
results = sarima_model.fit(disp=False)
results.summary()  # AIC/BIC, p-values, residual diagnostics

In [ ]:
# Obtain predicted values and plot the predictions
start = len(data_train)
end = len(data)

predictions = results.predict(
    start=start,                     # time to start
    end=end,                         # time to end
    dynamic=False,                   # use the full history to forecast
).rename(f'SARIMA{str(arima_params)}{str(seasonal_params)} Predictions')

pd.concat([data[["time", "co2"]], predictions], axis=1).plot(x="time", figsize=(12, 6));

## 6. Prophet

[Prophet](https://facebook.github.io/prophet/) is a decomposable forecasting model (trend + seasonality + holidays) developed by Meta. Unlike SARIMA, it handles non-linear trends and multiple seasonalities without requiring stationarity. It uses a piecewise linear or logistic growth curve and Fourier series for seasonality.

<div class="alert alert-success">
    <b>EXERCISE:</b>
     <ul>
      <li>Use Prophet to do the forecasting.</li>
    </ul>
</div>

Solution: `solutions/co2-3_forecasting-prophet.py`

In [ ]:
# !pip install -U prophet

## 7. Model comparison

In [ ]:
# ── Gather test-period predictions for each model ──────────────────────────────

test_time = data_test["time"].values

# 1. Linear + seasonal (sklearn pipeline fitted earlier as `pipe`)
pred_linear = pipe.predict(data_test[['year', 'month']])

# 2. SARIMA (statsmodels already fitted as `results`)
pred_sarima = results.predict(
    start=len(data_train),
    end=len(data) - 1,
    dynamic=False,
).values

# 3. Prophet (fitted above as `forecast_prophet`)
#    forecast_prophet rows beyond training length correspond to the test period.
pred_prophet = forecast_prophet["yhat"].iloc[-len(data_test):].values

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

# Show full history in grey for context
ax.plot(data["time"], data["co2"], color="lightgray", linewidth=0.8, label="Observed")

# Highlight test window
ax.plot(test_time, data_test["co2"].values, color="black", linewidth=1.5, label="Test (actual)")

ax.plot(test_time, pred_linear,  label="Linear + seasonal",  linestyle="--")
ax.plot(test_time, pred_sarima,  label="SARIMA",             linestyle="-.")
ax.plot(test_time, pred_prophet, label="Prophet",            linestyle=":")

ax.axvline(2018, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("Year")
ax.set_ylabel("CO₂ (ppm)")
ax.set_title("Model comparison on test set (2018 onward)")
ax.legend()
plt.tight_layout()

## Going further

- [statsmodels time-series analysis](https://www.statsmodels.org/dev/tsa.html) — ARIMA, VAR, state-space models

- [pmdarima docs](https://alkaline-ml.com/pmdarima/) — sklearn-compatible ARIMA- [Prophet docs](https://facebook.github.io/prophet/)